# 01 EDA: Heart Disease Dataset

This notebook documents the shared dataset used by the project team. It covers the data source, feature types, missing values, duplicates, class balance, feature distributions, and correlation analysis.

The test split is inspected only for basic integrity checks such as shape, columns, missing values, and target balance. It must not be used for model selection, hyperparameter tuning, feature selection, threshold selection, or fitting preprocessing statistics.

## Dataset Source

The project uses the 2024 Behavioral Risk Factor Surveillance System (BRFSS) dataset published by the U.S. Centers for Disease Control and Prevention (CDC). The raw data are distributed in SAS Transport (`.XPT`) format and can be downloaded with `data/raw/download_data.py`.

The prediction target is binary heart disease status. It is constructed from two BRFSS variables: `CVDINFR4` and `CVDCRHD4`. These original variables are removed after target construction to avoid target leakage.

In [ ]:
import os
from pathlib import Path
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data import load_splits, TARGET_COLUMN
from src.preprocessing import (
    NUMERICAL_FEATURES,
    BINARY_FEATURES,
    CATEGORICAL_FEATURES,
    fit_preprocessor,
    transform_features,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
train_df, validation_df, test_df = load_splits()

splits = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

pd.DataFrame(
    {
        split_name: {
            "objects": len(df),
            "features_without_target": df.shape[1] - 1,
            "total_columns": df.shape[1],
        }
        for split_name, df in splits.items()
    }
).T

## Feature Groups

In [ ]:
feature_groups = pd.DataFrame(
    [
        {"feature": feature, "group": "numerical"}
        for feature in NUMERICAL_FEATURES
    ]
    + [
        {"feature": feature, "group": "binary"}
        for feature in BINARY_FEATURES
    ]
    + [
        {"feature": feature, "group": "categorical"}
        for feature in CATEGORICAL_FEATURES
    ]
)

feature_groups

## Data Types

In [ ]:
pd.DataFrame(
    {
        split_name: df.dtypes.astype(str)
        for split_name, df in splits.items()
    }
)

## Missing Values

In [ ]:
missing_summary = []
for split_name, df in splits.items():
    missing = df.isna().sum()
    missing_summary.append(
        pd.DataFrame(
            {
                "split": split_name,
                "feature": missing.index,
                "missing_count": missing.values,
                "missing_share": missing.values / len(df),
            }
        )
    )

missing_summary = pd.concat(missing_summary, ignore_index=True)
missing_summary[missing_summary["missing_count"] > 0].sort_values(
    ["split", "missing_count"], ascending=[True, False]
)

## Duplicates

The cleaning script removes duplicate respondents using `_STATE` and `SEQNO` before selecting modeling features. The checks below count exact duplicate modeling rows after feature selection and imputation, where different respondents can have identical feature values.

In [ ]:
pd.DataFrame(
    {
        split_name: {
            "exact_duplicate_rows": df.duplicated().sum(),
            "exact_duplicate_share": df.duplicated().mean(),
        }
        for split_name, df in splits.items()
    }
).T

## Class Balance

In [ ]:
class_balance = []
for split_name, df in splits.items():
    counts = df[TARGET_COLUMN].value_counts().sort_index()
    shares = df[TARGET_COLUMN].value_counts(normalize=True).sort_index()
    class_balance.append(
        pd.DataFrame(
            {
                "split": split_name,
                "target": counts.index,
                "count": counts.values,
                "share": shares.values,
            }
        )
    )

class_balance = pd.concat(class_balance, ignore_index=True)
class_balance

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=class_balance, x="split", y="share", hue="target", ax=ax)
ax.set_title("Class balance by split")
ax.set_xlabel("")
ax.set_ylabel("Share")
ax.legend(title="Target")
plt.tight_layout()

## Numerical Feature Distributions

Distribution plots are based on the training set, because modeling decisions should be made without looking at the test set.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, feature in zip(axes, NUMERICAL_FEATURES):
    sns.histplot(data=train_df, x=feature, hue=TARGET_COLUMN, bins=40, stat="density", common_norm=False, ax=ax)
    ax.set_title(feature)

plt.tight_layout()

## Categorical and Binary Feature Distributions

In [ ]:
plot_features = BINARY_FEATURES + CATEGORICAL_FEATURES
n_cols = 3
n_rows = -(-len(plot_features) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.ravel()

for ax, feature in zip(axes, plot_features):
    order = sorted(train_df[feature].dropna().unique())
    sns.countplot(data=train_df, x=feature, hue=TARGET_COLUMN, order=order, ax=ax)
    ax.set_title(feature)
    ax.tick_params(axis="x", rotation=30)

for ax in axes[len(plot_features):]:
    ax.axis("off")

plt.tight_layout()

## Correlation Matrix

This correlation matrix uses the training set only. Categorical variables are still represented by their cleaned integer codes here, so correlations involving nominal categories should be interpreted cautiously.

In [ ]:
corr = train_df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="vlag", center=0, linewidths=0.3, ax=ax)
ax.set_title("Correlation matrix, train split")
plt.tight_layout()

## Preprocessing Check

The shared preprocessing object one-hot encodes categorical features, passes binary features through unchanged, and scales numerical features. It is fitted on train only.

In [ ]:
preprocessor = fit_preprocessor(train_df, scale_numerical=True)

X_train_preprocessed = transform_features(train_df, preprocessor)
X_validation_preprocessed = transform_features(validation_df, preprocessor)

pd.DataFrame(
    {
        "dataset": ["train", "validation"],
        "objects": [len(X_train_preprocessed), len(X_validation_preprocessed)],
        "preprocessed_features": [X_train_preprocessed.shape[1], X_validation_preprocessed.shape[1]],
    }
)